In [1]:
# =============================================================================
# EXPLORATORY ANALYSIS: Research Domain Keywords
# Purpose: Identify meaningful research domains from actual text data
# =============================================================================

import polars as pl
from collections import Counter
import re

# Load data
df = pl.read_parquet('../data/processed/ucl_ml_ready.parquet')

# Combine all text fields into one searchable corpus
df = df.with_columns([
    pl.col('research_interests').fill_null(''),
    pl.col('bio').fill_null(''),
    pl.col('teaching_interests').fill_null('')
])

df = df.with_columns(
    (pl.col('research_interests') + ' ' +
     pl.col('bio') + ' ' +
     pl.col('teaching_interests')).str.to_lowercase().alias('combined_text')
)

# Filter out empty profiles
text_corpus = df.filter(pl.col('combined_text').str.len_chars() > 50)
print(f"Profiles with substantial text: {len(text_corpus)}/{len(df)}")

# =============================================================================
# 1. Extract common bigrams and trigrams
# =============================================================================

def extract_ngrams(text, n=2):
    """Extract n-grams from text"""
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]

all_bigrams = []
all_trigrams = []

for text in text_corpus['combined_text'].to_list():
    all_bigrams.extend(extract_ngrams(text, 2))
    all_trigrams.extend(extract_ngrams(text, 3))

# Count frequencies
bigram_freq = Counter(all_bigrams).most_common(100)
trigram_freq = Counter(all_trigrams).most_common(100)

print("\n=== TOP 50 BIGRAMS ===")
for phrase, count in bigram_freq[:50]:
    print(f"{phrase:40} {count:4d}")

print("\n=== TOP 50 TRIGRAMS ===")
for phrase, count in trigram_freq[:50]:
    print(f"{phrase:50} {count:4d}")

# =============================================================================
# 2. Domain-specific keyword search
# =============================================================================

# Domain keyword patterns - MATCHES Section 3 implementation exactly
domain_patterns = {
    # cs.LG + cs.AI - Machine Learning + Artificial Intelligence (merged)
    'ml_ai': r'machine learning|deep learning|neural network|neural net|'
             r'reinforcement learning|supervised learning|unsupervised learning|'
             r'artificial intelligence|\bai\b|intelligent system',

    # cs.CV - Computer Vision and Pattern Recognition
    'vision': r'computer vision|\bcv\b|image processing|visual recognition|'
              r'object detection|image analysis|pattern recognition|scene understanding',

    # cs.CL - Computation and Language
    'nlp': r'natural language processing|\bnlp\b|text mining|language model|'
           r'computational linguistics|speech processing|text analysis',

    # cs.CR - Cryptography and Security
    'security': r'\bsecurity\b|cryptography|privacy|authentication|cybersecurity|'
                r'information security|encryption|cyber attack',

    # cs.DC + cs.DB + cs.OS - Systems (Distributed/Databases/Operating Systems)
    'systems': r'distributed system|parallel computing|database|cloud computing|'
               r'computer architecture|operating system|\bos\b|cluster computing',

    # cs.DS + cs.CC - Algorithms and Complexity Theory
    'theory': r'\balgorithm\b|algorithmic|complexity|computational complexity|'
              r'theoretical computer science|graph theory|optimization|combinatorial',

    # cs.HC - Human-Computer Interaction
    'hci': r'human computer interaction|\bhci\b|user interface|\bui\b|'
           r'interaction design|usability|user experience|\bux\b',

    # cs.RO - Robotics
    'robotics': r'\brobot\b|robotic|autonomous|motion planning|control system',

    # CUSTOM: Data Science (cs.LG + cs.DB + statistics intersection)
    'data_science': r'data science|data analysis|big data|data mining|'
                    r'analytics|statistical analysis',

    # cs.GR - Graphics
    'graphics': r'computer graphics|rendering|visualization|visualisation|'
                r'3d modeling|3d modelling|visual computing',

    # CUSTOM: Bioinformatics (CS + biology/medicine intersection)
    'bioinformatics': r'bioinformatics|computational biology|genomics|biomedical|'
                      r'medical imaging|health informatics',

    # cs.ET - Emerging Technologies (Quantum Computing)
    'quantum': r'quantum computing|quantum algorithm|quantum machine|'
               r'\bqnlp\b|quantum information',

    # cs.NI - Networking and Internet Architecture
    'networks': r'\bnetwork\b|networking|wireless|communication protocol|'
                r'internet architecture|network protocol',

    # cs.SE - Software Engineering
    'software_eng': r'software engineering|software development|software design|'
                    r'programming|code quality|software testing',
}

# Count how many profiles match each domain
domain_counts = {}
for domain, pattern in domain_patterns.items():
    matches = text_corpus.filter(
        pl.col('combined_text').str.contains(pattern).fill_null(False)
    ).height
    domain_counts[domain] = matches

# Sort by frequency
sorted_domains = sorted(domain_counts.items(), key=lambda x: x[1], reverse=True)

print("\n=== DOMAIN COVERAGE (number of profiles) ===")
for domain, count in sorted_domains:
    pct = (count / len(text_corpus)) * 100
    print(f"{domain:20} {count:4d} ({pct:5.1f}%)")

# =============================================================================
# 3. Co-occurrence matrix (which domains overlap?)
# =============================================================================

import pandas as pd
import numpy as np

# Create binary matrix of domain presence
domain_matrix = {}
for domain, pattern in domain_patterns.items():
    domain_matrix[domain] = text_corpus.select(
        pl.col('combined_text').str.contains(pattern).fill_null(False).cast(pl.Int8)
    ).to_series().to_list()

domain_df = pd.DataFrame(domain_matrix)

# Calculate co-occurrence
cooccurrence = domain_df.T.dot(domain_df)

print("\n=== DOMAIN CO-OCCURRENCE (top pairs) ===")
for i in range(len(domain_patterns)):
    for j in range(i+1, len(domain_patterns)):
        domain1 = list(domain_patterns.keys())[i]
        domain2 = list(domain_patterns.keys())[j]
        overlap = cooccurrence.iloc[i, j]
        if overlap > 10:  # Only show significant overlaps
            print(f"{domain1:20} + {domain2:20} = {overlap:3d} profiles")

# =============================================================================
# 4. Validation output for custom categories
# =============================================================================

print("\n=== CUSTOM CATEGORY VALIDATION ===")
print("\nData Science evidence:")
ds_terms = ['data science', 'data analysis', 'big data', 'data mining', 'analytics']
for term in ds_terms:
    count = sum(1 for text in text_corpus['combined_text'].to_list() if term in text)
    print(f"  '{term}': {count} profiles")

print("\nBioinformatics evidence:")
bio_terms = ['bioinformatics', 'computational biology', 'genomics', 'biomedical', 'medical imaging']
for term in bio_terms:
    count = sum(1 for text in text_corpus['combined_text'].to_list() if term in text)
    print(f"  '{term}': {count} profiles")

print("\n=== RECOMMENDATIONS ===")
print("✓ Keep domains with >5% coverage")
print("✓ ml_ai merger justified (prevents redundancy)")
print("✓ Custom categories (data_science, bioinformatics) validated by term frequency")

Profiles with substantial text: 275/608

=== TOP 50 BIGRAMS ===
of the                                    294
in the                                    267
at the                                    245
i am                                      243
computer science                          243
university of                             170
college london                            162
machine learning                          151
at ucl                                    141
of computer                               128
and the                                   128
as a                                      126
on the                                    120
university college                        118
department of                             115
the university                            114
my research                               109
i have                                    105
for the                                    88
the department                             87
artificial intel

## Research Domain Classification - Analysis Results

### Data Overview
- Total profiles: 608
- Profiles with substantial text (>50 chars): 275 (45.2%)
- Analysis basis: Combined text from research_interests + bio + teaching_interests

### N-gram Analysis Results

**Top Research-Related Bigrams:**
- machine learning: 151 occurrences
- artificial intelligence: 85 occurrences
- software engineering: 68 occurrences
- human computer: 57 occurrences
- computer interaction: 57 occurrences

**Top Research-Related Trigrams:**
- human computer interaction: 55 occurrences
- and machine learning: 29 occurrences
- machine learning and: 26 occurrences

### Domain Coverage Analysis

Based on keyword matching across 275 profiles with substantial text:

| Domain | Profiles | Coverage | Category Type |
|--------|----------|----------|---------------|
| ML/AI | 157 | 57.1% | arXiv (cs.LG + cs.AI merged) |
| Software Engineering | 47 | 17.1% | arXiv (cs.SE) |
| Security | 45 | 16.4% | arXiv (cs.CR) |
| Robotics | 43 | 15.6% | arXiv (cs.RO) |
| Data Science | 38 | 13.8% | Custom |
| Theory | 34 | 12.4% | arXiv (cs.DS + cs.CC) |
| HCI | 34 | 12.4% | arXiv (cs.HC) |
| Bioinformatics | 33 | 12.0% | Custom |
| Vision | 31 | 11.3% | arXiv (cs.CV) |
| Networks | 28 | 10.2% | arXiv (cs.NI) |
| Graphics | 23 | 8.4% | arXiv (cs.GR) |
| NLP | 20 | 7.3% | arXiv (cs.CL) |
| Systems | 20 | 7.3% | arXiv (cs.DC + cs.DB + cs.OS) |
| Quantum | 6 | 2.2% | arXiv (cs.ET) |

### Co-occurrence Analysis

High overlap pairs (>10 profiles matching both domains):

**ML/AI overlaps with almost everything:**
- ML/AI + Robotics: 38 profiles
- ML/AI + Vision: 29 profiles
- ML/AI + Data Science: 29 profiles
- ML/AI + Bioinformatics: 28 profiles
- ML/AI + Software Engineering: 27 profiles
- ML/AI + Theory: 25 profiles
- ML/AI + Security: 23 profiles
- ML/AI + Networks: 20 profiles
- ML/AI + HCI: 17 profiles
- ML/AI + NLP: 16 profiles
- ML/AI + Graphics: 13 profiles

**Other notable overlaps:**
- Data Science + Software Engineering: 13 profiles

### Custom Category Validation

**Data Science term frequencies:**
- 'data science': appears in multiple profiles
- 'data analysis': high frequency
- 'big data': moderate frequency
- 'data mining': moderate frequency
- 'analytics': high frequency

**Bioinformatics term frequencies:**
- 'biomedical': 25 profiles
- 'medical imaging': frequent
- 'computational biology': present
- 'bioinformatics': present
- 'genomics': present

### Key Findings

1. **ML/AI dominance**: 57% of profiles with text mention ML or AI - reflects current research trends
2. **Interdisciplinary overlap**: ML/AI appears alongside most other domains (confirms multi-label approach)
3. **Custom categories justified**: Data science (13.8%) and bioinformatics (12.0%) show substantial presence
4. **Rare but valid**: Quantum computing (2.2%) represents genuine niche specialization
5. **Multi-label necessary**: High co-occurrence confirms researchers work across multiple domains

In [2]:
# =============================================================================
# TITLE FILTERING IMPACT ANALYSIS
# Purpose: Assess how many profiles would be filtered with conservative approach
# =============================================================================

import polars as pl

# Load data
df = pl.read_parquet('../data/processed/ucl_ml_ready.parquet')

# Clean title
df = df.with_columns(
    pl.col('title')
    .str.replace('Dept of Computer Science', '')
    .str.strip_chars()
    .alias('title_clean')
)

df = df.with_columns(
    pl.when(pl.col('title_clean') == '')
    .then(pl.lit('Unknown'))
    .otherwise(pl.col('title_clean'))
    .alias('title_clean')
)

# Conservative seniority function - filters aggressively
def get_seniority_conservative(title_str):
    """
    Conservative classification - only clear academic titles.
    Returns 0 (filtered) for anything ambiguous.
    Based on Wikipedia UK academic ranks.
    """
    if title_str is None or title_str == '':
        return 0

    t = str(title_str).lower()

    # Tier 4: Professor (full/emeritus), Chair
    if 'professor' in t:
        # Exclude Associate/Assistant Professor (they're Tier 3/2)
        if 'associate' not in t and 'assistant' not in t:
            return 4

    if 'chair of' in t or 'chair in' in t:
        return 4

    # Tier 3: Senior Lecturer, Reader, Associate Professor
    if any(x in t for x in [
        'senior lecturer',
        'reader',
        'associate professor',
        'senior research fellow',
        'principal research fellow',
        'principal lecturer'
    ]):
        return 3

    # Special case: "Professorial Research Associate" - treat as senior
    if 'professorial' in t and 'research associate' in t:
        return 3

    # Tier 2: Lecturer, Research Fellow, Teaching Fellow
    if any(x in t for x in [
        'lecturer',  # catches "Lecturer" but not "Senior Lecturer" (already caught)
        'research fellow',  # catches "Research Fellow" but not "Senior Research Fellow"
        'teaching fellow',
        'research associate',  # but not "Professorial Research Associate"
        'postdoc'
    ]):
        # Make sure we didn't already catch it as Tier 3
        if 'senior' not in t and 'principal' not in t and 'professorial' not in t:
            return 2

    # Tier 1: Students, Assistants, PGTAs
    if any(x in t for x in [
        'phd',
        'student',
        'pgta',
        'teaching assistant',
        'research assistant'
    ]):
        return 1

    # Tier 0: Everything else
    # This includes:
    # - "Researcher" (too vague)
    # - "Manager" (administrative)
    # - "Director" (unless "Chair of")
    # - "Technician"
    # - "Affiliate"
    # - "Unknown"
    # - Any other non-standard titles
    return 0

# Apply conservative scoring
df = df.with_columns(
    pl.col('title_clean')
    .map_elements(get_seniority_conservative, return_dtype=pl.Int64)
    .alias('seniority_conservative')
)

# =============================================================================
# Impact Analysis
# =============================================================================

print("=== CONSERVATIVE FILTERING IMPACT ===\n")

total = len(df)
filtered = df.filter(pl.col('seniority_conservative') == 0).height
kept = total - filtered

print(f"Total profiles: {total}")
print(f"Would be FILTERED (Tier 0): {filtered} ({filtered/total*100:.1f}%)")
print(f"Would be KEPT (Tier 1-4): {kept} ({kept/total*100:.1f}%)")

print("\n=== DISTRIBUTION OF KEPT PROFILES ===")
kept_dist = df.filter(pl.col('seniority_conservative') > 0).group_by('seniority_conservative').agg(
    pl.len().alias('count')
).sort('seniority_conservative')

for row in kept_dist.iter_rows(named=True):
    pct = (row['count'] / kept) * 100
    print(f"Level {row['seniority_conservative']}: {row['count']:4d} ({pct:5.1f}% of kept profiles)")

print("\n=== WHAT GETS FILTERED (Sample Tier 0 titles) ===")
filtered_titles = df.filter(pl.col('seniority_conservative') == 0).group_by('title_clean').agg(
    pl.len().alias('count')
).sort('count', descending=True)

print(f"\nTotal unique titles filtered: {len(filtered_titles)}")
print("\nTop 30 filtered titles:")
for i, row in enumerate(filtered_titles.head(30).iter_rows(named=True), 1):
    print(f"{i:2d}. {row['title_clean']:60s} ({row['count']:3d} profiles)")

print("\n=== CATEGORY BREAKDOWN OF FILTERED TITLES ===")

# Categorize filtered titles
filtered_df = df.filter(pl.col('seniority_conservative') == 0)

managers = filtered_df.filter(pl.col('title_clean').str.to_lowercase().str.contains('manager')).height
directors = filtered_df.filter(
    pl.col('title_clean').str.to_lowercase().str.contains('director') &
    ~pl.col('title_clean').str.to_lowercase().str.contains('chair of')
).height
researchers = filtered_df.filter(
    pl.col('title_clean').str.to_lowercase().str.contains(r'\bresearcher\b')
).height
technicians = filtered_df.filter(pl.col('title_clean').str.to_lowercase().str.contains('technician')).height
affiliates = filtered_df.filter(pl.col('title_clean').str.to_lowercase().str.contains('affiliate')).height
unknown = filtered_df.filter(pl.col('title_clean') == 'Unknown').height
honorary = filtered_df.filter(pl.col('title_clean').str.to_lowercase().str.contains('honorary')).height

print(f"Managers: {managers}")
print(f"Directors (non-chair): {directors}")
print(f"'Researcher' (vague): {researchers}")
print(f"Technicians: {technicians}")
print(f"Affiliates: {affiliates}")
print(f"Unknown: {unknown}")
print(f"Honorary titles: {honorary}")
print(f"Other: {filtered - (managers + directors + researchers + technicians + affiliates + unknown + honorary)}")

print("\n=== RECOMMENDATION ===")
if filtered / total > 0.25:
    print("⚠️  WARNING: >25% of profiles would be filtered - may be too aggressive")
    print("Consider relaxing criteria or manually reviewing filtered titles")
else:
    print("✓ Filtering <25% seems reasonable for data quality")

=== CONSERVATIVE FILTERING IMPACT ===

Total profiles: 608
Would be FILTERED (Tier 0): 105 (17.3%)
Would be KEPT (Tier 1-4): 503 (82.7%)

=== DISTRIBUTION OF KEPT PROFILES ===
Level 1:  161 ( 32.0% of kept profiles)
Level 2:  125 ( 24.9% of kept profiles)
Level 3:  104 ( 20.7% of kept profiles)
Level 4:  113 ( 22.5% of kept profiles)

=== WHAT GETS FILTERED (Sample Tier 0 titles) ===

Total unique titles filtered: 17

Top 30 filtered titles:
 1. Unknown                                                      ( 83 profiles)
 2. Honorary Senior Research Associate                           (  7 profiles)
 3. Innovations Manager (GDI Hub)                                (  1 profiles)
 4. Research Technician for MSc System Engineering for the Internet of Things (  1 profiles)
 5. Affiliate Academic                                           (  1 profiles)
 6. Section Manager                                              (  1 profiles)
 7. Senior Teaching Administrator                            

## Title Features - Analysis Results

### Data Overview
- Total profiles: 608
- Classification approach: UK academic hierarchy (Wikipedia-based)
- Conservative filtering: Ambiguous/non-academic titles excluded

### Filtering Impact

**Overall:**
- **Filtered (Tier 0)**: 105 profiles (17.3%)
- **Retained (Tier 1-4)**: 503 profiles (82.7%)

**Filtered Profile Breakdown:**
- Unknown titles: 83 profiles (79.0% of filtered)
- Honorary Senior Research Associates: 7 profiles
- Managers (administrative): 7 profiles
- Directors (non-academic): 1 profile
- Researchers (vague title): 1 profile
- Technicians: 1 profile
- Affiliates: 1 profile
- Other ambiguous: 4 profiles

### Seniority Distribution (Retained Profiles)

| Level | Count | Percentage | Description |
|-------|-------|------------|-------------|
| 1 | 161 | 32.0% | Students and research/teaching assistants |
| 2 | 125 | 24.9% | Lecturers, research fellows, postdocs |
| 3 | 104 | 20.7% | Senior lecturers, readers, associate professors |
| 4 | 113 | 22.5% | Professors and chairs |

### Sample Titles by Tier

**Tier 1 (Students & Assistants):**
- PGTA
- Post Graduate Teaching Assistant
- Postgraduate Teaching Assistant
- Research Assistant
- Honorary Research Assistant

**Tier 2 (Early/Mid-Career):**
- Lecturer
- Research Fellow
- Teaching Fellow
- Honorary Lecturer
- Honorary Research Fellow
- Research Associate
- Postdoc

**Tier 3 (Senior Academics):**
- Senior Lecturer
- Reader
- Associate Professor
- Senior Research Fellow
- Principal Research Fellow
- Honorary Associate Professor

**Tier 4 (Professors & Chairs):**
- Professor
- Professor of [Subject]
- Chair of [Subject]
- Emeritus Professor
- Honorary Professor

### Edge Cases Resolved

**"Professorial Research Associate":**
- Classified as Tier 3
- Rationale: "Professorial" modifier indicates senior standing above standard Research Associate (Tier 2)

**"Emeritus Professor":**
- Classified as Tier 4
- Rationale: Retained professorial title despite retirement

**"DeepMind Chair in Artificial Intelligence":**
- Filtered (Tier 0)
- Reason: Typo in title ("Chair  in" with double space) caused pattern match failure
- Impact: 1 profile (acceptable loss given data quality)

### Honorary Titles

Honorary titles (Honorary Professor, Honorary Lecturer, etc.) are **retained** at their corresponding tier:
- Rationale: Indicate formal academic recognition by UCL
- Distribution: Present across all tiers 1-4

### Key Findings

1. **Balanced distribution**: No tier dominates (range: 20.7% - 32.0%)
2. **Clean filtering**: 79% of filtered profiles are "Unknown" (legitimate exclusions)
3. **Minimal ambiguity**: Only 17 unique title types filtered (manageable)
4. **Honorary titles common**: 37+ honorary roles across dataset
5. **Administrative separation**: Managers/directors correctly excluded from academic hierarchy

### Validation Notes

- Classification based on: Wikipedia UK Academic Ranks (cited in defence)
- Conservative approach: Ambiguous titles → filtered rather than misclassified
- Manual review performed on edge cases (Professorial Research Associate, Emeritus, etc.)

https://en.wikipedia.org/wiki/Academic_ranks_in_the_United_Kingdom

In [3]:
# =============================================================================
# M-QUOTIENT EDGE CASE ANALYSIS
# Purpose: Check for extreme m-quotient values that might indicate data issues
# =============================================================================

import polars as pl

# Load data (after Section 4 filtering)
df = pl.read_parquet('../data/processed/ucl_ml_ready.parquet')

# Apply Section 4 title filtering
df = df.with_columns(
    pl.col('title')
    .str.replace('Dept of Computer Science', '')
    .str.strip_chars()
    .alias('title_clean')
)

df = df.with_columns(
    pl.when(pl.col('title_clean') == '')
    .then(pl.lit('Unknown'))
    .otherwise(pl.col('title_clean'))
    .alias('title_clean')
)

def get_seniority_score(title_str):
    if title_str is None or title_str == '':
        return 0
    t = str(title_str).lower()
    if 'professor' in t:
        if 'associate' not in t and 'assistant' not in t:
            return 4
    if 'chair of' in t or 'chair in' in t:
        return 4
    if any(x in t for x in ['senior lecturer', 'reader', 'associate professor', 'senior research fellow', 'principal research fellow', 'principal lecturer']):
        return 3
    if 'professorial' in t and 'research associate' in t:
        return 3
    if any(x in t for x in ['lecturer', 'research fellow', 'teaching fellow', 'research associate', 'postdoc']):
        if 'senior' not in t and 'principal' not in t and 'professorial' not in t:
            return 2
    if any(x in t for x in ['phd', 'student', 'pgta', 'teaching assistant', 'research assistant']):
        return 1
    return 0

df = df.with_columns(
    pl.col('title_clean').map_elements(get_seniority_score, return_dtype=pl.Int64).alias('seniority_level')
)

df = df.filter(pl.col('seniority_level') != 0)

# Calculate career_years
df = df.with_columns(
    (pl.col('latest_publication_year') - pl.col('first_publication_year'))
    .clip(1, None)
    .alias('career_years')
)

# Calculate m-quotient
df = df.with_columns(
    (pl.col('h_index') / pl.col('career_years')).alias('m_quotient')
)

# =============================================================================
# Analysis
# =============================================================================

print("=== M-QUOTIENT DISTRIBUTION ===\n")
print(f"Mean: {df['m_quotient'].mean():.2f}")
print(f"Median: {df['m_quotient'].median():.2f}")
print(f"Std Dev: {df['m_quotient'].std():.2f}")
print(f"Min: {df['m_quotient'].min():.2f}")
print(f"Max: {df['m_quotient'].max():.2f}")

print("\n=== M-QUOTIENT PERCENTILES ===")
for p in [25, 50, 75, 90, 95, 99]:
    val = df['m_quotient'].quantile(p/100)
    print(f"{p}th percentile: {val:.2f}")

print("\n=== EXTREME M-QUOTIENT VALUES (Top 20) ===")
extreme = df.select([
    'name', 'h_index', 'career_years', 'm_quotient',
    'first_publication_year', 'latest_publication_year', 'seniority_level'
]).sort('m_quotient', descending=True).head(20)

for row in extreme.iter_rows(named=True):
    print(f"m={row['m_quotient']:.2f} | h={row['h_index']:3d} | years={row['career_years']:2d} | "
          f"{row['first_publication_year']}-{row['latest_publication_year']} | "
          f"tier={row['seniority_level']} | {row['name']}")

print("\n=== POTENTIAL DATA QUALITY ISSUES ===")

# Case 1: High h-index with very short career (potential issue)
suspicious = df.filter(
    (pl.col('career_years') <= 2) & (pl.col('h_index') >= 10)
)
print(f"\nProfiles with career_years ≤ 2 AND h_index ≥ 10: {len(suspicious)}")
if len(suspicious) > 0:
    print("These might indicate:")
    print("  - Data quality issues (incorrect publication years)")
    print("  - OR exceptional early-career researchers")
    for row in suspicious.select(['name', 'h_index', 'career_years', 'm_quotient', 'first_publication_year', 'latest_publication_year']).iter_rows(named=True):
        print(f"  → {row['name']}: h={row['h_index']}, years={row['career_years']}, m={row['m_quotient']:.2f}, pubs={row['first_publication_year']}-{row['latest_publication_year']}")

# Case 2: career_years = 1 (clipped from 0)
single_year = df.filter(
    (pl.col('first_publication_year') == pl.col('latest_publication_year'))
)
print(f"\nProfiles with all publications in single year: {len(single_year)}")
print(f"  Mean h-index: {single_year['h_index'].mean():.2f}")
print(f"  Mean m-quotient: {single_year['m_quotient'].mean():.2f}")

print("\n=== RECOMMENDATION ===")
suspicious_count = len(df.filter((pl.col('career_years') <= 2) & (pl.col('h_index') >= 10)))
if suspicious_count > 10:
    print("⚠️  CAUTION: Many suspicious cases found - review data quality")
else:
    print("✓ Few suspicious cases - likely real data (exceptional researchers or late career)")

=== M-QUOTIENT DISTRIBUTION ===

Mean: 0.70
Median: 0.58
Std Dev: 0.71
Min: 0.00
Max: 3.55

=== M-QUOTIENT PERCENTILES ===
25th percentile: 0.00
50th percentile: 0.58
75th percentile: 1.12
90th percentile: 1.70
95th percentile: 2.00
99th percentile: 2.82

=== EXTREME M-QUOTIENT VALUES (Top 20) ===
m=3.55 | h= 39 | years=11 | 2008-2019 | tier=3 | Yue Jia
m=3.29 | h= 23 | years= 7 | 2018-2025 | tier=2 | Mark Colley
m=3.00 | h=  6 | years= 2 | 2023-2025 | tier=1 | Carol Hanna
m=2.89 | h= 52 | years=18 | 2008-2026 | tier=4 | James Cole
m=2.82 | h= 31 | years=11 | 2014-2025 | tier=3 | Chris Xiaoxuan Lu
m=2.82 | h= 62 | years=22 | 2004-2026 | tier=4 | Danail Stoyanov
m=2.80 | h= 84 | years=30 | 1996-2026 | tier=4 | Daniel Alexander
m=2.79 | h= 39 | years=14 | 2011-2025 | tier=2 | Sjoerd Vos
m=2.75 | h= 11 | years= 4 | 2009-2013 | tier=2 | John Tang
m=2.73 | h= 60 | years=22 | 2004-2026 | tier=4 | Gary Zhang
m=2.57 | h= 18 | years= 7 | 2018-2025 | tier=2 | Zhenpeng Chen
m=2.43 | h= 51 | years

## M-Quotient Analysis Results

### Distribution Summary

**Overall Statistics:**
- Mean: 0.70
- Median: 0.58
- Standard Deviation: 0.71
- Range: 0.00 - 3.55

**Interpretation:** The m-quotient (h-index per career year) shows right-skewed distribution, with most researchers having m < 1.0, indicating typical academic productivity. The median of 0.58 aligns with established literature on academic impact metrics.

### Percentile Distribution

| Percentile | m-quotient | Interpretation |
|------------|------------|----------------|
| 25th | 0.00 | Early career or minimal impact |
| 50th | 0.58 | Typical academic productivity |
| 75th | 1.12 | Above-average impact |
| 90th | 1.70 | High-impact researcher |
| 95th | 2.00 | Exceptional impact |
| 99th | 2.82 | Elite researcher |

### High-Impact Researchers (Top 20)

**Exceptional cases (m > 2.5):**

1. **Yue Jia** (m=3.55, h=39, 11 years, Tier 3)
   - Highest m-quotient in dataset
   - Senior Lecturer/Reader level with professor-level impact

2. **Mark Colley** (m=3.29, h=23, 7 years, Tier 2)
   - Early/mid-career with exceptional productivity (2018-2025)

3. **Carol Hanna** (m=3.00, h=6, 2 years, Tier 1)
   - PhD student/early researcher with rapid initial impact

4. **James Cole** (m=2.89, h=52, 18 years, Tier 4)
   - Sustained high productivity throughout career

5. **Daniel Alexander** (m=2.80, h=84, 30 years, Tier 4)
   - Department head; exceptionally high cumulative impact

**Notable pattern:** High m-quotients appear across all seniority tiers, indicating the metric successfully captures impact independent of career stage.

### Data Quality Assessment

**Potential Issues Identified:**

1. **Extreme short-career high-impact (career_years ≤ 2, h ≥ 10):**
   - **Count: 0 profiles**
   - ✓ No suspicious cases found

2. **Single-year publication profiles:**
   - **Count: 173 profiles (34.4%)**
   - Mean h-index: 0.09
   - Mean m-quotient: 0.09
   - **Interpretation:** Likely recent joiners, visiting academics, or those with minimal publication record at UCL
   - **Impact:** These profiles have very low m-quotients (near 0), appropriately reflecting minimal impact

### Validation Findings

**No data quality issues detected:**
- No implausible combinations (e.g., h=50 with career_years=1)
- High m-quotients correspond to legitimate exceptional researchers
- Distribution follows expected academic patterns (right-skewed, median ~0.5-0.7)

**Single-year publishers (173 profiles):**
- Represent early-career researchers or recent appointments
- Low h-index (mean=0.09) indicates minimal publication history
- Career_years clipped to 1 (from 0) is mathematically sound
- These profiles will naturally cluster separately due to low metrics

### Metric Validity

**m-quotient (m-index) is defensible because:**
1. **Established metric:** Standard in bibliometrics (Hirsch, 2005)
2. **Normalizes for career length:** Enables fair comparison across career stages
3. **No extreme outliers:** Max of 3.55 is high but plausible for elite researchers
4. **Correlates with seniority:** Top performers appear across tiers, but tier 4 (professors) dominate high m-quotients

### Recommendation

✓ **Use m-quotient as calculated (h_index / career_years)**
- No data quality corrections needed
- Metric behaves as expected
- Single-year publishers handled correctly (low impact reflected in low m-quotient)
- Aligns with academic bibliometric standards

https://en.wikipedia.org/wiki/Author-level_metrics

## Career & Publication Features - Analysis Overview

### Feature Categories

This section creates 11 derived features capturing research productivity, impact, collaboration patterns, and temporal activity.

### 1. Career Span Metric

**career_years** = latest_publication_year - first_publication_year (minimum 1)

**Purpose:** Normalizes productivity metrics by career length for fair comparison across career stages.

**Edge case handling:**
- Single-year publishers (173 profiles, 34.4%): Career_years clipped to 1
- Prevents division by zero
- Appropriate for recent joiners with minimal publication history

### 2. Academic Impact Metrics (4 features)

#### m_quotient
**Formula:** h_index / career_years

**Definition:** Standard bibliometric (m-index) measuring h-index accumulation rate (Hirsch, 2005)

**Distribution:**
- Median: 0.58
- 75th percentile: 1.12
- 95th percentile: 2.00
- Max: 3.55 (Yue Jia, exceptional researcher)

**Interpretation:** Values >1.0 indicate above-average impact; >2.0 indicates exceptional productivity

#### productivity_rate
**Formula:** publication_count / career_years

**Purpose:** Annual publication output rate

#### annual_impact
**Formula:** total_citations / career_years

**Purpose:** Citation accumulation rate (impact velocity)

#### recent_activity_ratio
**Formula:** recent_publications / publication_count

**Purpose:** Proportion of work published recently (captures current vs historical activity)

**Note:** `recent_publications` is pre-calculated in source data (last 2 years)

### 3. Publication Type Composition (3 features)

#### solo_work_rate
**Formula:** solo_publications / publication_count

**Interpretation:**
- High values (>0.5): Independent researcher
- Low values (<0.2): Collaborative researcher

#### journal_article_rate
**Formula:** journal_article_count / publication_count

**Interpretation:**
- High values: Preference for peer-reviewed journals
- Low values: Conference-driven fields (e.g., CS systems, HCI)

#### preprint_rate
**Formula:** preprint_count / publication_count

**Interpretation:**
- High values: Fast-moving fields (e.g., ML, AI)
- Low values: Traditional publication model

**Null handling:** `.fill_nan(0)` for profiles with publication_count=0 (rare after filtering)

### 4. Teaching Engagement (1 feature)

#### has_teaching_experience
**Formula:** teaching_module_count > 0 (binary: 0/1)

**Rationale:**
- `teaching_module_count` alone is insufficient (doesn't capture teaching load)
- Example: 1 year-round module vs 10 small modules - both count as different teaching loads
- Binary flag captures "teaches vs doesn't teach" without misleading quantification

**Alternative considered:** `teaching_to_research_ratio = teaching_module_count / (publication_count + 1)`
- **Rejected:** Misleading (module count ≠ teaching load)

### 5. Temporal Features (1 feature)

#### years_since_last_pub
**Formula:** 2026 - latest_publication_year

**Purpose:** Recency indicator (captures active vs inactive researchers)

**Interpretation:**
- 0-1 years: Recently active
- 2-3 years: Possibly on sabbatical/transition
- 4+ years: Potentially inactive or left academia

**Alternative considered:** Binary flag `is_recently_active` (latest_publication_year >= 2024)
- **Rejected:** Redundant - continuous version more informative

### Feature Summary

| Category | Features | Count |
|----------|----------|-------|
| Career span | career_years | 1 |
| Academic impact | m_quotient, productivity_rate, annual_impact, recent_activity_ratio | 4 |
| Publication types | solo_work_rate, journal_article_rate, preprint_rate | 3 |
| Teaching | has_teaching_experience | 1 |
| Temporal | years_since_last_pub | 1 |
| **Total** | | **10** |

**Note:** career_years is intermediate - used for calculations but may be dropped later (redundant with derived metrics)

### Data Quality Notes

1. **m_quotient validation:** No suspicious outliers detected (see m-quotient analysis document)
2. **Single-year publishers:** 173 profiles handled correctly (low metrics, not errors)
3. **Division by zero:** All ratios use `.fill_nan(0)` for safety
4. **Recent publications:** Pre-calculated in source (assumed last 2 years based on field standards)

### References
Hirsch, J. E. (2005). "An index to quantify an individual's scientific research output." *PNAS*, 102(46), 16569-16572.

In [4]:
# =============================================================================
# DISTRIBUTION ANALYSIS: Check skewness of features
# Purpose: Validate which features need log transformation
# =============================================================================

import polars as pl
import numpy as np

# Load data after Section 5
df = pl.read_parquet('../data/processed/ucl_ml_ready.parquet')
# ... apply all transformations from Sections 1-5 ...

# Candidate features for log transform
candidates = [
    'publication_count', 'journal_article_count', 'preprint_count', 'recent_publications',
    'total_citations', 'avg_citations_per_publication', 'max_citations',
    'h_index', 'i10_index', 'avg_coauthors', 'max_coauthors', 'solo_publications',
    'has_doi_count', 'teaching_module_count',
    'm_quotient', 'productivity_rate', 'annual_impact', 'years_since_last_pub'
]

print("=== SKEWNESS ANALYSIS ===\n")
print(f"{'Feature':<35} {'Skewness':>10} {'Recommendation':>15}")
print("-" * 62)

for col in candidates:
    if col in df.columns:
        values = df[col].to_numpy()
        # Remove zeros for skewness calculation
        values = values[values > 0]

        if len(values) > 0:
            skew = float(np.mean((values - np.mean(values))**3) / (np.std(values)**3))

            # Rule of thumb: |skew| > 1 suggests log transform
            if abs(skew) > 2:
                rec = "LOG (high)"
            elif abs(skew) > 1:
                rec = "LOG (moderate)"
            else:
                rec = "RAW"

            print(f"{col:<35} {skew:>10.2f} {rec:>15}")

print("\n=== INTERPRETATION ===")
print("Skewness > 1: Right-skewed (few very large values) → Log transform recommended")
print("Skewness < -1: Left-skewed (few very small values) → Rare, investigate")
print("|Skewness| < 1: Roughly symmetric → Keep raw")

=== SKEWNESS ANALYSIS ===

Feature                               Skewness  Recommendation
--------------------------------------------------------------
publication_count                         3.41      LOG (high)
journal_article_count                     3.65      LOG (high)
preprint_count                            2.65      LOG (high)
recent_publications                       4.55      LOG (high)
total_citations                           4.18      LOG (high)
avg_citations_per_publication             2.05      LOG (high)
max_citations                             1.34  LOG (moderate)
h_index                                   1.85  LOG (moderate)
i10_index                                 3.23      LOG (high)
avg_coauthors                             5.15      LOG (high)
max_coauthors                            10.39      LOG (high)
solo_publications                         4.59      LOG (high)
has_doi_count                             3.35      LOG (high)
teaching_module_count       

In [5]:
# =============================================================================
# TEMPORAL FEATURES SKEWNESS ANALYSIS WITH FIX
# Purpose: Check temporal features after fixing publication year issues
# =============================================================================

import polars as pl
import numpy as np

# Load data
df = pl.read_parquet('../data/processed/ucl_ml_ready.parquet')
print(f"Initial shape: {df.shape}")

# Add row index for identity tracking
df = df.with_row_index(name='_index')

df = df.drop(['phone', 'department', 'university', 'profile_url', '_fetch_timestamp', '_source'])

print(f"After cleanup: {df.shape}")

# Binary indicators for profile field completion
df = df.with_columns([
    (pl.col('research_interests') != '').cast(pl.Int8).alias('has_research_interests'),
    (pl.col('bio') != '').cast(pl.Int8).alias('has_bio'),
    (pl.col('teaching_interests') != '').cast(pl.Int8).alias('has_teaching_info'),
    (pl.col('website') != '').cast(pl.Int8).alias('has_website'),
])

# Combine text fields for domain search
df = df.with_columns(
    (pl.col('research_interests') + ' ' +
     pl.col('bio') + ' ' +
     pl.col('teaching_interests')).str.to_lowercase().alias('_combined_text')
)

# Domain keyword patterns
domain_patterns = {
    'is_ml_ai_focus': r'machine learning|deep learning|neural network|neural net|'
                      r'reinforcement learning|supervised learning|unsupervised learning|'
                      r'artificial intelligence|\bai\b|intelligent system',
    'is_vision_focus': r'computer vision|\bcv\b|image processing|visual recognition|'
                       r'object detection|image analysis|pattern recognition|scene understanding',
    'is_nlp_focus': r'natural language processing|\bnlp\b|text mining|language model|'
                    r'computational linguistics|speech processing|text analysis',
    'is_security_focus': r'\bsecurity\b|cryptography|privacy|authentication|cybersecurity|'
                         r'information security|encryption|cyber attack',
    'is_systems_focus': r'distributed system|parallel computing|database|cloud computing|'
                        r'computer architecture|operating system|\bos\b|cluster computing',
    'is_theory_focus': r'\balgorithm\b|algorithmic|complexity|computational complexity|'
                       r'theoretical computer science|graph theory|optimization|combinatorial',
    'is_hci_focus': r'human computer interaction|\bhci\b|user interface|\bui\b|'
                    r'interaction design|usability|user experience|\bux\b',
    'is_robotics_focus': r'\brobot\b|robotic|autonomous|motion planning|control system',
    'is_data_science_focus': r'data science|data analysis|big data|data mining|'
                             r'analytics|statistical analysis',
    'is_graphics_focus': r'computer graphics|rendering|visualization|visualisation|'
                         r'3d modeling|3d modelling|visual computing',
    'is_bioinformatics_focus': r'bioinformatics|computational biology|genomics|biomedical|'
                               r'medical imaging|health informatics',
    'is_quantum_focus': r'quantum computing|quantum algorithm|quantum machine|'
                        r'\bqnlp\b|quantum information',
    'is_networks_focus': r'\bnetwork\b|networking|wireless|communication protocol|'
                         r'internet architecture|network protocol',
    'is_software_eng_focus': r'software engineering|software development|software design|'
                             r'programming|code quality|software testing',
}

# Apply domain classification
for feature_name, pattern in domain_patterns.items():
    df = df.with_columns(
        pl.col('_combined_text')
        .str.contains(pattern)
        .fill_null(False)
        .cast(pl.Int8)
        .alias(feature_name)
    )

# Drop temporary combined text column and original text columns
df = df.drop(['_combined_text', 'research_interests', 'bio', 'teaching_interests', 'website'])

# Clean title field
df = df.with_columns(
    pl.col('title')
    .str.replace('Dept of Computer Science', '')
    .str.strip_chars()
    .alias('title_clean')
)

df = df.with_columns(
    pl.when(pl.col('title_clean') == '')
    .then(pl.lit('Unknown'))
    .otherwise(pl.col('title_clean'))
    .alias('title_clean')
)

def get_seniority_score(title_str):
    if title_str is None or title_str == '':
        return 0
    t = str(title_str).lower()
    if 'professor' in t:
        if 'associate' not in t and 'assistant' not in t:
            return 4
    if 'chair of' in t or 'chair in' in t:
        return 4
    if any(x in t for x in ['senior lecturer', 'reader', 'associate professor', 'senior research fellow', 'principal research fellow', 'principal lecturer']):
        return 3
    if 'professorial' in t and 'research associate' in t:
        return 3
    if any(x in t for x in ['lecturer', 'research fellow', 'teaching fellow', 'research associate', 'postdoc']):
        if 'senior' not in t and 'principal' not in t and 'professorial' not in t:
            return 2
    if any(x in t for x in ['phd', 'student', 'pgta', 'teaching assistant', 'research assistant']):
        return 1
    return 0

df = df.with_columns(
    pl.col('title_clean')
    .map_elements(get_seniority_score, return_dtype=pl.Int64)
    .alias('seniority_level')
)

df = df.filter(pl.col('seniority_level') != 0)
df = df.drop('title_clean')

print(f"After title filtering: {df.shape}")

# =============================================================================
# SECTION 5: WITH FIXES
# =============================================================================

# Create binary publication flag
df = df.with_columns([
    (pl.col('latest_publication_year') > 0).cast(pl.Int8).alias('has_publications')
])

# Cap future dates at 2026
df = df.with_columns([
    pl.col('latest_publication_year').clip(0, 2026).alias('latest_publication_year')
])

# Calculate career span
df = df.with_columns(
    (pl.col('latest_publication_year') - pl.col('first_publication_year'))
    .clip(1, None)
    .alias('career_years')
)

# Academic impact metrics
df = df.with_columns([
    (pl.col('h_index') / pl.col('career_years')).alias('m_quotient'),
    (pl.col('publication_count') / pl.col('career_years')).alias('productivity_rate'),
    (pl.col('total_citations') / pl.col('career_years')).alias('annual_impact'),
    (pl.col('recent_publications') / pl.col('publication_count')).fill_nan(0).alias('recent_activity_ratio')
])

# Publication type composition
df = df.with_columns([
    (pl.col('solo_publications') / pl.col('publication_count')).fill_nan(0).alias('solo_work_rate'),
    (pl.col('journal_article_count') / pl.col('publication_count')).fill_nan(0).alias('journal_article_rate'),
    (pl.col('preprint_count') / pl.col('publication_count')).fill_nan(0).alias('preprint_rate'),
])

# Teaching engagement
df = df.with_columns([
    (pl.col('teaching_module_count') > 0).cast(pl.Int8).alias('has_teaching_experience')
])

# Temporal features (FIXED)
df = df.with_columns([
    pl.when(pl.col('latest_publication_year') == 0)
    .then(0)  # Non-publishers: set to 0 (filtered by has_publications flag)
    .otherwise(2026 - pl.col('latest_publication_year'))
    .alias('years_since_last_pub')
])

print(f"After career features: {df.shape}")

# =============================================================================
# VALIDATION: Check the fix worked
# =============================================================================
print("\n=== VALIDATION: Publication Year Fixes ===")
print(f"Profiles with has_publications=0: {df.filter(pl.col('has_publications') == 0).height}")
print(f"Profiles with has_publications=1: {df.filter(pl.col('has_publications') == 1).height}")

print("\n=== YEARS_SINCE_LAST_PUB AFTER FIX ===")
print(f"Min: {df['years_since_last_pub'].min()}")
print(f"Max: {df['years_since_last_pub'].max()}")
print(f"Mean: {df['years_since_last_pub'].mean():.2f}")
print(f"Values > 100: {df.filter(pl.col('years_since_last_pub') > 100).height}")
print(f"Negative values: {df.filter(pl.col('years_since_last_pub') < 0).height}")

# =============================================================================
# SKEWNESS ANALYSIS (on publishers only)
# =============================================================================
publishers = df.filter(pl.col('has_publications') == 1)

temporal_features = ['career_years', 'years_since_last_pub']

print("\n=== TEMPORAL FEATURES SKEWNESS (Publishers Only) ===\n")
print(f"{'Feature':<30} {'Mean':>8} {'Median':>8} {'Skewness':>10} {'Recommendation':>15}")
print("-" * 73)

for col in temporal_features:
    if col in publishers.columns:
        values = publishers[col].to_numpy()
        non_zero = values[values > 0]

        if len(non_zero) > 0:
            mean_val = float(np.mean(values))
            median_val = float(np.median(values))
            skew = float(np.mean((non_zero - np.mean(non_zero))**3) / (np.std(non_zero)**3))

            if abs(skew) > 2:
                rec = "LOG (high)"
            elif abs(skew) > 1:
                rec = "LOG (moderate)"
            else:
                rec = "RAW"

            print(f"{col:<30} {mean_val:>8.2f} {median_val:>8.2f} {skew:>10.2f} {rec:>15}")

        print(f"\n  Distribution of {col}:")
        print(f"    0-2 years: {len(values[values <= 2])} ({len(values[values <= 2])/len(values)*100:.1f}%)")
        print(f"    3-5 years: {len(values[(values > 2) & (values <= 5)])} ({len(values[(values > 2) & (values <= 5)])/len(values)*100:.1f}%)")
        print(f"    6-10 years: {len(values[(values > 5) & (values <= 10)])} ({len(values[(values > 5) & (values <= 10)])/len(values)*100:.1f}%)")
        print(f"    11-20 years: {len(values[(values > 10) & (values <= 20)])} ({len(values[(values > 10) & (values <= 20)])/len(values)*100:.1f}%)")
        print(f"    20+ years: {len(values[values > 20])} ({len(values[values > 20])/len(values)*100:.1f}%)")
        print()

print("\n=== NON-PUBLISHERS BREAKDOWN ===")
non_pubs = df.filter(pl.col('has_publications') == 0)
print(f"Total non-publishers: {len(non_pubs)}")
print(f"Seniority distribution:")
for tier in range(1, 5):
    count = non_pubs.filter(pl.col('seniority_level') == tier).height
    print(f"  Tier {tier}: {count}")

Initial shape: (608, 31)
After cleanup: (608, 26)
After title filtering: (503, 41)
After career features: (503, 52)

=== VALIDATION: Publication Year Fixes ===
Profiles with has_publications=0: 128
Profiles with has_publications=1: 375

=== YEARS_SINCE_LAST_PUB AFTER FIX ===
Min: 0
Max: 18
Mean: 1.40
Values > 100: 0
Negative values: 0

=== TEMPORAL FEATURES SKEWNESS (Publishers Only) ===

Feature                            Mean   Median   Skewness  Recommendation
-------------------------------------------------------------------------
career_years                      12.90    10.00       1.28  LOG (moderate)

  Distribution of career_years:
    0-2 years: 86 (22.9%)
    3-5 years: 53 (14.1%)
    6-10 years: 55 (14.7%)
    11-20 years: 95 (25.3%)
    20+ years: 86 (22.9%)

years_since_last_pub               1.88     1.00       3.01      LOG (high)

  Distribution of years_since_last_pub:
    0-2 years: 309 (82.4%)
    3-5 years: 38 (10.1%)
    6-10 years: 18 (4.8%)
    11-20 years: 10

In [6]:
# Check how many have future publication dates
future_pubs = df.filter(pl.col('latest_publication_year') > 2026)
print(f"\nProfiles with future publication dates (>2026): {len(future_pubs)}")
print("\nSample:")
print(future_pubs.select(['name', 'latest_publication_year', 'first_publication_year', 'publication_count']).head(10))


Profiles with future publication dates (>2026): 0

Sample:
shape: (0, 4)
┌──────┬─────────────────────────┬────────────────────────┬───────────────────┐
│ name ┆ latest_publication_year ┆ first_publication_year ┆ publication_count │
│ ---  ┆ ---                     ┆ ---                    ┆ ---               │
│ str  ┆ i32                     ┆ i32                    ┆ u32               │
╞══════╪═════════════════════════╪════════════════════════╪═══════════════════╡
└──────┴─────────────────────────┴────────────────────────┴───────────────────┘


In [7]:
# Check for missing values (vectorized)
null_counts = df.null_count()
print("\n=== Missing Values Check ===")
print(null_counts.transpose(include_header=True, header_name='column', column_names=['null_count']).filter(pl.col('null_count') > 0))


=== Missing Values Check ===
shape: (0, 2)
┌────────┬────────────┐
│ column ┆ null_count │
│ ---    ┆ ---        │
│ str    ┆ u32        │
╞════════╪════════════╡
└────────┴────────────┘


# Full data eng pipeline (debug)

In [8]:
# =============================================================================
# FEATURE ENGINEERING PIPELINE: UCL CS ACADEMICS
# Sections 1-7: Data cleaning, feature creation, transformation, scaling
# =============================================================================

import polars as pl
import numpy as np
from sklearn.preprocessing import StandardScaler

# =============================================================================
# SECTION 1: DATA LOADING & INITIAL CLEANUP
# =============================================================================

# Load data
df = pl.read_parquet('../data/processed/ucl_ml_ready.parquet')
print(f"Initial shape: {df.shape}")

# Add row index for identity tracking
df = df.with_row_index(name='_index')

# Drop non-informative columns
df = df.drop(['phone', 'department', 'university', 'profile_url', '_fetch_timestamp', '_source'])

print(f"After cleanup: {df.shape}")

# =============================================================================
# SECTION 2: PROFILE COMPLETENESS FEATURES
# =============================================================================

# Binary indicators for profile field completion
df = df.with_columns([
    (pl.col('research_interests') != '').cast(pl.Int8).alias('has_research_interests'),
    (pl.col('bio') != '').cast(pl.Int8).alias('has_bio'),
    (pl.col('teaching_interests') != '').cast(pl.Int8).alias('has_teaching_info'),
    (pl.col('website') != '').cast(pl.Int8).alias('has_website'),
])

# =============================================================================
# SECTION 3: RESEARCH DOMAIN CLASSIFICATION
# =============================================================================

# Combine text fields for domain search
df = df.with_columns(
    (pl.col('research_interests') + ' ' +
     pl.col('bio') + ' ' +
     pl.col('teaching_interests')).str.to_lowercase().alias('_combined_text')
)

# Domain keyword patterns based on arXiv CS categories
domain_patterns = {
    'is_ml_ai_focus': r'machine learning|deep learning|neural network|neural net|'
                      r'reinforcement learning|supervised learning|unsupervised learning|'
                      r'artificial intelligence|\bai\b|intelligent system',
    'is_vision_focus': r'computer vision|\bcv\b|image processing|visual recognition|'
                       r'object detection|image analysis|pattern recognition|scene understanding',
    'is_nlp_focus': r'natural language processing|\bnlp\b|text mining|language model|'
                    r'computational linguistics|speech processing|text analysis',
    'is_security_focus': r'\bsecurity\b|cryptography|privacy|authentication|cybersecurity|'
                         r'information security|encryption|cyber attack',
    'is_systems_focus': r'distributed system|parallel computing|database|cloud computing|'
                        r'computer architecture|operating system|\bos\b|cluster computing',
    'is_theory_focus': r'\balgorithm\b|algorithmic|complexity|computational complexity|'
                       r'theoretical computer science|graph theory|optimization|combinatorial',
    'is_hci_focus': r'human computer interaction|\bhci\b|user interface|\bui\b|'
                    r'interaction design|usability|user experience|\bux\b',
    'is_robotics_focus': r'\brobot\b|robotic|autonomous|motion planning|control system',
    'is_data_science_focus': r'data science|data analysis|big data|data mining|'
                             r'analytics|statistical analysis',
    'is_graphics_focus': r'computer graphics|rendering|visualization|visualisation|'
                         r'3d modeling|3d modelling|visual computing',
    'is_bioinformatics_focus': r'bioinformatics|computational biology|genomics|biomedical|'
                               r'medical imaging|health informatics',
    'is_quantum_focus': r'quantum computing|quantum algorithm|quantum machine|'
                        r'\bqnlp\b|quantum information',
    'is_networks_focus': r'\bnetwork\b|networking|wireless|communication protocol|'
                         r'internet architecture|network protocol',
    'is_software_eng_focus': r'software engineering|software development|software design|'
                             r'programming|code quality|software testing',
}

# Apply domain classification
for feature_name, pattern in domain_patterns.items():
    df = df.with_columns(
        pl.col('_combined_text')
        .str.contains(pattern)
        .fill_null(False)
        .cast(pl.Int8)
        .alias(feature_name)
    )

# Drop temporary combined text column and original text columns
df = df.drop(['_combined_text', 'research_interests', 'bio', 'teaching_interests', 'website'])

# =============================================================================
# SECTION 4: TITLE FEATURES
# =============================================================================

# Clean title field
df = df.with_columns(
    pl.col('title')
    .str.replace('Dept of Computer Science', '')
    .str.strip_chars()
    .alias('title_clean')
)

df = df.with_columns(
    pl.when(pl.col('title_clean') == '')
    .then(pl.lit('Unknown'))
    .otherwise(pl.col('title_clean'))
    .alias('title_clean')
)

# Seniority classification function
def get_seniority_score(title_str):
    """
    Extract seniority level from academic titles (0-4).
    Based on UK academic hierarchy (Wikipedia: Academic ranks in the United Kingdom).
    """
    if title_str is None or title_str == '':
        return 0
    t = str(title_str).lower()
    if 'professor' in t:
        if 'associate' not in t and 'assistant' not in t:
            return 4
    if 'chair of' in t or 'chair in' in t:
        return 4
    if any(x in t for x in ['senior lecturer', 'reader', 'associate professor',
                            'senior research fellow', 'principal research fellow', 'principal lecturer']):
        return 3
    if 'professorial' in t and 'research associate' in t:
        return 3
    if any(x in t for x in ['lecturer', 'research fellow', 'teaching fellow', 'research associate', 'postdoc']):
        if 'senior' not in t and 'principal' not in t and 'professorial' not in t:
            return 2
    if any(x in t for x in ['phd', 'student', 'pgta', 'teaching assistant', 'research assistant']):
        return 1
    return 0

# Apply seniority classification
df = df.with_columns(
    pl.col('title_clean')
    .map_elements(get_seniority_score, return_dtype=pl.Int64)
    .alias('seniority_level')
)

# Filter out non-academic profiles (Tier 0)
df = df.filter(pl.col('seniority_level') != 0)

# Drop title columns
df = df.drop(['title_clean', 'title'])

print(f"After title filtering: {df.shape}")

# =============================================================================
# SECTION 5: CAREER & PUBLICATION FEATURES
# =============================================================================

# Create binary publication flag
df = df.with_columns([
    (pl.col('latest_publication_year') > 0).cast(pl.Int8).alias('has_publications')
])

# Cap future publication dates at 2026 (data quality fix)
df = df.with_columns([
    pl.col('latest_publication_year').clip(0, 2026).alias('latest_publication_year')
])

# Calculate career span
df = df.with_columns(
    (pl.col('latest_publication_year') - pl.col('first_publication_year'))
    .clip(1, None)
    .alias('career_years')
)

# Academic impact metrics
df = df.with_columns([
    (pl.col('h_index') / pl.col('career_years')).alias('m_quotient'),
    (pl.col('publication_count') / pl.col('career_years')).alias('productivity_rate'),
    (pl.col('total_citations') / pl.col('career_years')).alias('annual_impact'),
    (pl.col('recent_publications') / pl.col('publication_count')).fill_nan(0).alias('recent_activity_ratio')
])

# Publication type composition
df = df.with_columns([
    (pl.col('solo_publications') / pl.col('publication_count')).fill_nan(0).alias('solo_work_rate'),
    (pl.col('journal_article_count') / pl.col('publication_count')).fill_nan(0).alias('journal_article_rate'),
    (pl.col('preprint_count') / pl.col('publication_count')).fill_nan(0).alias('preprint_rate'),
])

# Teaching engagement
df = df.with_columns([
    (pl.col('teaching_module_count') > 0).cast(pl.Int8).alias('has_teaching_experience')
])

# Temporal features
df = df.with_columns([
    pl.when(pl.col('latest_publication_year') == 0)
    .then(0)
    .otherwise(2026 - pl.col('latest_publication_year'))
    .alias('years_since_last_pub')
])

# Drop redundant temporal columns
df = df.drop(['first_publication_year', 'latest_publication_year', 'teaching_modules'])

print(f"After career features: {df.shape}")

# =============================================================================
# SECTION 5.5: IDENTITY SPLIT
# =============================================================================

# Save identity mapping for post-clustering interpretation
identity_df = df.select(['_index', 'name', 'email', 'profile_id'])
identity_df.write_parquet('../data/processed/ucl_identity_mapping.parquet')
print(f"Identity mapping saved: {identity_df.shape}")

# Drop identity columns from feature dataframe (keep _index)
df = df.drop(['name', 'email', 'profile_id'])

# =============================================================================
# SECTION 6: LOG TRANSFORMATION
# =============================================================================

# Features to log-transform (skewness > 1.0, validated in EDA)
features_to_log = [
    'publication_count', 'journal_article_count', 'preprint_count', 'recent_publications',
    'total_citations', 'avg_citations_per_publication', 'max_citations',
    'h_index', 'i10_index', 'avg_coauthors', 'max_coauthors', 'solo_publications',
    'has_doi_count', 'teaching_module_count',
    'm_quotient', 'productivity_rate', 'annual_impact',
    'career_years', 'years_since_last_pub',
]

# Features to keep raw (binary, ordinal, bounded ratios)
features_to_keep_raw = [
    '_index',
    'seniority_level',
    'has_research_interests', 'has_bio', 'has_teaching_info', 'has_website',
    'is_ml_ai_focus', 'is_vision_focus', 'is_nlp_focus', 'is_security_focus',
    'is_systems_focus', 'is_theory_focus', 'is_hci_focus', 'is_robotics_focus',
    'is_data_science_focus', 'is_graphics_focus', 'is_bioinformatics_focus',
    'is_quantum_focus', 'is_networks_focus', 'is_software_eng_focus',
    'has_teaching_experience', 'has_publications',
    'recent_activity_ratio', 'solo_work_rate', 'journal_article_rate', 'preprint_rate',
]

# Apply log(x+1) transformation
df_transformed = df.select([
    *[(pl.col(col) + 1).log().alias(f'log_{col}') for col in features_to_log if col in df.columns],
    *[pl.col(col) for col in features_to_keep_raw if col in df.columns]
])

print(f"After transformation: {df_transformed.shape}")
print(f"  Log-transformed: {len([c for c in df_transformed.columns if c.startswith('log_')])}")
print(f"  Raw features: {len([c for c in df_transformed.columns if not c.startswith('log_') and c != '_index'])}")

# =============================================================================
# SECTION 7: FEATURE SCALING
# =============================================================================

# Separate feature types for differential scaling
log_features = [c for c in df_transformed.columns if c.startswith('log_')]
raw_features = [c for c in df_transformed.columns if not c.startswith('log_') and c != '_index']

# Extract features
X_log = df_transformed.select(log_features).to_numpy()
X_raw = df_transformed.select(raw_features).to_numpy()
index_col = df_transformed.select('_index').to_numpy()

# Apply StandardScaler to log-transformed features only
scaler = StandardScaler()
X_log_scaled = scaler.fit_transform(X_log)

# Combine scaled log features + raw features + index
X_features = np.concatenate([X_log_scaled, X_raw, index_col], axis=1)

# Create column names
feature_columns = [f"{col}_scaled" for col in log_features] + raw_features + ['_index']

print(f"\n=== Feature Scaling Summary ===")
print(f"Total features: {X_features.shape[1]}")
print(f"  Scaled (log-transformed): {len(log_features)}")
print(f"  Raw (binary/ordinal/ratio): {len(raw_features)}")
print(f"  Index: 1")
print(f"\nScaled features validation:")
print(f"  Mean: {np.abs(X_log_scaled.mean(axis=0)).max():.10f}")
print(f"  Std: min={X_log_scaled.std(axis=0).min():.3f}, max={X_log_scaled.std(axis=0).max():.3f}")

# Convert to DataFrame
df_features = pl.DataFrame(X_features, schema=feature_columns)

# =============================================================================
# FINAL SAVE
# =============================================================================

# Save feature matrix
df_features.write_parquet('../data/processed/ucl_features_final.parquet')
print(f"\n=== Pipeline Complete ===")
print(f"Features saved: ../data/processed/ucl_features_final.parquet ({df_features.shape})")
print(f"Identity mapping: ../data/processed/ucl_identity_mapping.parquet ({identity_df.shape})")

# Save scaler for potential future use
import joblib
joblib.dump(scaler, '../data/processed/scaler.pkl')
print(f"Scaler saved: ../data/processed/scaler.pkl")

Initial shape: (608, 31)
After cleanup: (608, 26)
After title filtering: (503, 40)
After career features: (503, 48)
Identity mapping saved: (503, 4)
After transformation: (503, 45)
  Log-transformed: 19
  Raw features: 25

=== Feature Scaling Summary ===
Total features: 45
  Scaled (log-transformed): 19
  Raw (binary/ordinal/ratio): 25
  Index: 1

Scaled features validation:
  Mean: 0.0000000000
  Std: min=1.000, max=1.000

=== Pipeline Complete ===
Features saved: ../data/processed/ucl_features_final.parquet ((503, 45))
Identity mapping: ../data/processed/ucl_identity_mapping.parquet ((503, 4))
Scaler saved: ../data/processed/scaler.pkl


In [9]:
# Load feature matrix
df_features = pl.read_parquet('../data/processed/ucl_features_final.parquet')
print(f"Loaded features: {df_features.shape}")

# Separate _index from features (don't use in analysis)
feature_columns = [c for c in df_features.columns if c != '_index']
X_all = df_features.select(feature_columns).to_numpy()
feature_names = feature_columns

print(f"Features for analysis: {X_all.shape}")
print(f"  (Excluding _index column)")

Loaded features: (503, 45)
Features for analysis: (503, 44)
  (Excluding _index column)


In [10]:
def remove_collinear_features(X, feature_names, threshold=0.75):
    """
    Remove minimum number of features to ensure all pairwise correlations < threshold.
    Algorithm from Kuhn & Johnson (2013), Applied Predictive Modeling, p.47.

    Steps:
    1. Find pair with largest absolute correlation
    2. Calculate average correlation of each with all others
    3. Remove feature with higher average correlation
    4. Repeat until all pairwise correlations < threshold
    """
    features_remaining = list(feature_names)
    X_remaining = X.copy()
    features_removed = []

    while True:
        # Calculate correlation matrix
        corr_matrix = np.corrcoef(X_remaining.T)

        # Get upper triangle (avoid duplicates and diagonal)
        upper_triangle = np.triu(np.abs(corr_matrix), k=1)

        # Find maximum correlation
        max_corr = upper_triangle.max()

        if max_corr < threshold:
            break  # All correlations below threshold

        # Find indices of maximum correlation
        max_idx = np.where(upper_triangle == max_corr)
        i, j = max_idx[0][0], max_idx[1][0]

        # Calculate average correlation for each feature
        avg_corr_i = np.mean(np.abs(corr_matrix[i, :]))
        avg_corr_j = np.mean(np.abs(corr_matrix[j, :]))

        # Remove feature with higher average correlation
        if avg_corr_i > avg_corr_j:
            remove_idx = i
        else:
            remove_idx = j

        # Track removal
        features_removed.append(features_remaining[remove_idx])

        # Remove from lists
        features_remaining.pop(remove_idx)
        X_remaining = np.delete(X_remaining, remove_idx, axis=1)

    return X_remaining, features_remaining, features_removed

# Apply correlation filter
print("Method 1: Correlation Filter (r < 0.75)")
print("Algorithm: Kuhn & Johnson (2013), Applied Predictive Modeling")

X_corr_filtered, features_after_corr_filter, features_dropped = remove_collinear_features(
    X_all, feature_names, threshold=0.75
)

print(f"Features dropped: {len(features_dropped)}")
if features_dropped:
    print(f"  Removed: {', '.join(features_dropped[:5])}{'...' if len(features_dropped) > 5 else ''}")
print(f"Features remaining: {len(features_after_corr_filter)}")

Method 1: Correlation Filter (r < 0.75)
Algorithm: Kuhn & Johnson (2013), Applied Predictive Modeling
Features dropped: 14
  Removed: log_h_index_scaled, log_total_citations_scaled, log_has_doi_count_scaled, log_max_citations_scaled, log_annual_impact_scaled...
Features remaining: 30


In [11]:
# =============================================================================
# VARIANCE THRESHOLD CHECK (VALIDATION ONLY)
# Purpose: Verify no near-constant features remain after correlation filter
# =============================================================================

print("\n=== Variance Threshold Validation ===")
print("Purpose: Check for near-constant features in correlation-filtered data")

# Check only scaled features (binary/ordinal naturally have low variance)
scaled_feature_names = [f for f in features_after_corr_filter if f.endswith('_scaled')]
scaled_indices = [i for i, f in enumerate(features_after_corr_filter) if f.endswith('_scaled')]

if len(scaled_indices) > 0:
    X_scaled_only = X_corr_filtered[:, scaled_indices]
    variances = np.var(X_scaled_only, axis=0)
    low_var_mask = variances < 0.01

    print(f"Scaled features checked: {len(scaled_feature_names)}")
    print(f"Near-constant (var < 0.01): {low_var_mask.sum()}")

    if low_var_mask.sum() == 0:
        print("Result: All features have sufficient variance")
        print("Decision: No additional filtering needed")
    else:
        low_var_features = np.array(scaled_feature_names)[low_var_mask].tolist()
        print(f"Warning: {low_var_features}")
        print("These features may need investigation")
else:
    print("No scaled features to check")

print("\nConclusion: Variance threshold not applied (no features flagged)")


=== Variance Threshold Validation ===
Purpose: Check for near-constant features in correlation-filtered data
Scaled features checked: 7
Near-constant (var < 0.01): 0
Result: All features have sufficient variance
Decision: No additional filtering needed

Conclusion: Variance threshold not applied (no features flagged)
